In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import itertools
import os
import sys
sys.path.insert(0, "/".join(["..","python"]))

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from gridr.core.convolution.fft_filtering import (
    fft_array_filter,
    BoundarySpec,
    ConvolutionMethod,
    OutputMode,
    DecimationOrigin,
)
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im, mpl_plot_wrapper
IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
import notebook_utils
notebook_utils.FORCE_MPL_INTERACTIVE=True
if not IN_DOC_BUILD and not notebook_utils.FORCE_MPL_INTERACTIVE:
    from bokeh.io import output_notebook # enables plot interface in J notebook
    output_notebook()
from typing import get_args

@mpl_plot_wrapper
def plot_mesh3d_static(x, y, data, *, height=350, width=350, prefix=None):
    fig = plt.figure(figsize=(width / 100, height / 100), dpi=100)
    ax = fig.add_subplot(111, projection="3d") # Use original sizing strategy
    finite_data = data[np.isfinite(data)]
    
    if finite_data.size > 0:
        vmin, vmax = np.min(finite_data), np.max(finite_data)
    else:
        vmin, vmax = 0, 1
        
    X, Y = np.meshgrid(x, y)   
    ax.plot_wireframe(X, Y, data, rstride=1, cstride=1, color='blue', linewidth=0.3)
    plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
    
    return fig
    
def gaussian_kernel_2d(shape, sigma):
    """
    Generates a normalized 2D Gaussian kernel.
    
    :param shape: Kernel size (int for a square kernel, or tuple (height, width))
    :param sigma: Standard deviation of the Gaussian (float or tuple (sigma_y, sigma_x))
    :return: a tuple containing :
        - 1D Numpy array representing x coordinates
        - 1D Numpy array representing y coordinates
        - 2D NumPy array representing the kernel
    """
    # Handle scalar or tuple inputs for shape and sigma
    if isinstance(shape, int):
        h = w = shape
    else:
        h, w = shape
        
    if isinstance(sigma, (int, float)):
        sigma_y = sigma_x = sigma
    else:
        sigma_y, sigma_x = sigma

    # Create centered coordinates (e.g., size 5 -> [-2, -1, 0, 1, 2])
    y = np.arange(-(h // 2), h // 2 + 1 if h % 2 != 0 else h // 2)
    x = np.arange(-(w // 2), w // 2 + 1 if w % 2 != 0 else w // 2)
    
    # 2D Grid
    x_grid, y_grid = np.meshgrid(x, y)
    
    # Calculate 2D Gaussian function
    kernel = np.exp(-0.5 * ((x_grid / sigma_x)**2 + (y_grid / sigma_y)**2))
    
    # Normalize so the sum of all elements equals 1
    kernel_sum = np.sum(kernel)
    if kernel_sum != 0:
        kernel /= kernel_sum
        
    return x, y, kernel

def generate_high_frequency_test_image(size=512, frequency_factor=150):
    """
    Generates a 2D zone plate (chirp) image containing high spatial frequencies.
    
    :param size: Image resolution (size x size)
    :param frequency_factor: Controls how rapidly the frequency increases towards the edges
    :return: 2D NumPy array representing the test image
    """
    x = np.linspace(-1, 1, size)
    y = np.linspace(-1, 1, size)
    X, Y = np.meshgrid(x, y)

    # Frequency increases proportionally to the squared distance from the center
    R2 = X**2 + Y**2
    return 0.5 + 0.5 * np.sin(frequency_factor * R2)

    
def _window_rect(window, **kwargs):
    """Build a matplotlib Rectangle from an inclusive window ((r0, r1), (c0, c1))."""
    (r0, r1), (c0, c1) = window
    return Rectangle((c0 - 0.5, r0 - 0.5), c1 - c0 + 1, r1 - r0 + 1,
                     fill=False, **kwargs)


def valid_window_from_prod(window_prod, kernel_shape):
    """Compute the valid window (inclusive, full-image frame) from the same window.

    The kernel is assumed to be padded to an odd size: K_odd = K + (1 - K % 2).
    Returns None if the valid region is empty along any axis (N < K_odd).
    """
    window_prod = np.asarray(window_prod)
    k = np.asarray(kernel_shape[:2])
    radius = (k + (1 - k % 2)) // 2
    window_valid = window_prod + np.stack([radius, -radius], axis=1)
    if np.any(window_valid[:, 1] < window_valid[:, 0]):
        return None
    return window_valid

def _add_image_axes(fig, img, left, bottom, fig_w, fig_h, scale, **imshow_kw):
    """Add an axes displaying img at native resolution, positioned in screen pixels."""
    h, w = img.shape[:2]
    ax = fig.add_axes([left / fig_w, bottom / fig_h,
                       w * scale / fig_w, h * scale / fig_h])
    ax.imshow(img, cmap="gray", interpolation="none", resample=False,
              aspect="equal", **imshow_kw)
    # imshow puts pixel centers on integers, so pixel edges sit at +/- 0.5
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(h - 0.5, -0.5)
    ax.set_axis_off()
    return ax


def _crop(image, window):
    (r0, r1), (c0, c1) = window
    return image[r0:r1 + 1, c0:c1 + 1]


@mpl_plot_wrapper
def plot_image_output_mode(image_full, window_prod, kernel_shape, prefix=None,
                           scale=1, dpi=100):
    """Plot the full output with its same/valid regions, then each region alone.

    Each image pixel is rendered as exactly scale x scale screen pixels
    (scale must be an integer), with no interpolation. Panels are vertically
    centered on each other and linked by arrows.
    """
    window_prod = np.asarray(window_prod)
    window_valid = valid_window_from_prod(window_prod, kernel_shape)
    h, w = image_full.shape[:2]

    # Panels: (image, window in full coordinates or None, rect style)
    panels = [(image_full, None, None),
              (_crop(image_full, window_prod), window_prod,
               dict(edgecolor="tab:blue", linestyle="-", label="same"))]
    if window_valid is not None:
        window_valid = np.asarray(window_valid)
        panels.append((_crop(image_full, window_valid), window_valid,
                       dict(edgecolor="tab:green", linestyle="--", label="valid")))

    # Layout in screen pixels (integers to stay on the pixel grid)
    m_left, m_right, m_top, m_bottom = 10, 10, 24, 30
    gap = 40  # horizontal room for the arrows
    ax_h = h * scale  # the full image is the tallest panel
    widths = [p[0].shape[1] * scale for p in panels]
    fig_w = m_left + sum(widths) + gap * (len(panels) - 1) + m_right
    fig_h = ax_h + m_top + m_bottom

    fig = plt.figure(figsize=(fig_w / dpi, fig_h / dpi), dpi=dpi)
    norm_kw = dict(vmin=np.nanmin(image_full), vmax=np.nanmax(image_full))

    left = m_left
    y_center = m_bottom + ax_h / 2
    handles = []
    for i, (img, window, style) in enumerate(panels):
        p_h = img.shape[0] * scale
        bottom = m_bottom + (ax_h - p_h) // 2
        ax = _add_image_axes(fig, img, left, bottom, fig_w, fig_h, scale, **norm_kw)

        if i == 0:
            # Full panel: overlay every region
            handles.append(_window_rect([[0, h - 1], [0, w - 1]],
                                        edgecolor="tab:red", linewidth=1,
                                        linestyle="-", label="full"))
            handles += [_window_rect(win, linewidth=1, **st)
                        for _, win, st in panels[1:]]
            for patch in handles:
                ax.add_patch(patch)
        else:
            # Cropped panel: frame it with its region color
            ph, pw = img.shape[:2]
            ax.add_patch(_window_rect([[0, ph - 1], [0, pw - 1]],
                                      linewidth=1, **style))
            # Arrow from the previous panel
            x0 = left - gap + 6
            x1 = left - 6
            fig.add_artist(FancyArrowPatch(
                (x0 / fig_w, y_center / fig_h), (x1 / fig_w, y_center / fig_h),
                transform=fig.transFigure, arrowstyle="-|>",
                mutation_scale=10, linewidth=1, color="0.3"))

        left += widths[i] + gap

    win_h, win_w = window_prod[:, 1] - window_prod[:, 0] + 1
    fig.text(0.5, 1 - 4 / fig_h,
             f"full ({h}×{w}) - input ({win_h}×{win_w}) - "
             f"kernel ({kernel_shape[0]}×{kernel_shape[1]})",
             ha="center", va="top", fontsize=8)
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0),
               ncol=len(handles), frameon=False, fontsize=7)
    return fig

mandrill_0 = mandrill[0]
# The `gaussian_kernel_2d` computes the gaussian kernel - it implements the above mentionned formula.
gaussian_sigma = 4.
x, y, kernel = gaussian_kernel_2d(shape=int(gaussian_sigma*6+1), sigma=gaussian_sigma)

# Boundary conditions and output mode

This second tutorial describes how ``fft_array_filter`` handles the image borders, and how to choose which part of the convolution result is returned.

**What you'll learn**

- How the ``boundary`` argument extrapolates the values required outside the image domain
- How ``boundary`` and ``out_mode`` interact, and why the output size depends on both
- Why only the valid region is independent of the boundary condition
- How to interpret the returned ``window``, always expressed in the coordinates of the full output

Near the image borders, the convolution kernel extends outside the image domain and requires values that do not exist in the input.

`fft_array_filter` provides the `boundary` argument. It is used to set how these values are extrapolated.
It supports the modes already used by `numpy.pad` ("reflect", "symmetric", "edge", "constant" and "wrap") as well as "none", which applies no extrapolation.

The Fourier-domain backends available in GridR (`scipy.signal.oaconvolve` and `scipy.signal.fftconvolve`) do not provide a boundary condition parameter and implicitly extrapolate with zeros.
When `boundary` is not ``"none"``, `fft_array_filter` pads the input with the requested mode before calling the backend.

As a consequence, `boundary` and the `output_mode` parameter are closely related. The output mode ("full", "same" or "valid") defines which part of the convolution result is returned, and this part is computed from the padded array passed to the backend.
For a given `output_mode`, the size of the output therefore depends on the boundary condition.

For this section we still use the mandrill single channel image, but we limit its view to a smaller image in order to help visualize the behaviour at edges.

In [ ]:
mandrill_0_cropped = mandrill_0[0:192, 160:352]

We first use `boundary="none"`. In this case the input is not modified and the result is identical to a direct call to the SciPy backend.

In [ ]:
filtered_result_none_full = fft_array_filter(
        arr=mandrill_0_cropped,
        kernel=kernel,
        win=None,
        boundary="none",
        out_mode="full",
        zoom=(1, 1),
        decimation="leading",
        axes=None,
        dtype=None,
        method="overlap_add",
    )

In [ ]:
_ = plot_image_output_mode(filtered_result_none_full.data, filtered_result_none_full.window, kernel.shape, prefix=None)

In [ ]:
# direct call to scipy.signal.fftconvolve
from scipy.signal import fftconvolve
np.testing.assert_allclose(
    fftconvolve(mandrill_0_cropped, kernel, axes=None),
    filtered_result_none_full.data
)

In this first example the kernel is the same as the one used in the first section. Its size is 25x25, so the convolution requires a margin of 12 pixels on each side of the image. With a 192x192 input, the full output is therefore 216x216, which matches the result above.

Limiting the display to the "same" region also shows that the area between the same and valid windows is affected by the zeros used to extrapolate the image.

We now use `boundary="symmetric"`.

In [ ]:
filtered_result_symmetric_full=fft_array_filter(
        arr=mandrill_0_cropped,
        kernel=kernel,
        win=None,
        boundary="symmetric",
        out_mode="full",
        zoom=(1, 1),
        decimation="leading",
        axes=None,
        dtype=None,
        method="overlap_add",
)

In [ ]:
_ = plot_image_output_mode(filtered_result_symmetric_full.data, filtered_result_symmetric_full.window, kernel.shape, prefix=None)

Compared with the previous result, the margin between the full and same windows is now twice as large. The input is first padded by `fft_array_filter` with the kernel radius (12 pixels), and the SciPy backend then adds its own zero extrapolation of the same width.

The area between the same and valid windows is no longer affected by zeros, but it still depends on the extrapolated values and therefore on the chosen boundary condition. As
shown above, the result may look more acceptable, although this is subjective and depends largely on the input image.

Only the valid window is independent of the boundary condition, and its content is (almost) identical to the one obtained with `boundary="none"` :

In [ ]:
valid_slice_symmetric = (slice(36,-36), slice(36,-36))
valid_slice_none = (slice(24,-24), slice(24,-24))
np.testing.assert_allclose(filtered_result_symmetric_full.data[valid_slice_symmetric], filtered_result_none_full.data[valid_slice_none], rtol=1e-12)

The other boundary conditions lead to the same conclusions and are not detailed here.

The current implementation relies only on SciPy backends. For FFT-based methods, this means that part of the computation is spent on margins that are not needed, since SciPy always adds its own zero extrapolation on top of the padding applied by GridR.
These extra margins are usually small compared with the size of the images being processed, but they still represent unnecessary computation.

We now look more closely at the returned `window`. As mentioned before, it gives the position of the production window (the whole input image by default) within the filtered image.

This window is always expressed in the coordinates of the full output. When `output_mode` is not "full", it therefore cannot be applied directly to the returned data.

In [ ]:
# out_mode set to "same"
filtered_result_symmetric_same=fft_array_filter(
        arr=mandrill_0_cropped,
        kernel=kernel,
        win=None,
        boundary="symmetric",
        out_mode="same",
        zoom=(1, 1),
        decimation="leading",
        axes=None,
        dtype=None,
        method="overlap_add",
)

# out_mode set to "same"
filtered_result_symmetric_valid=fft_array_filter(
        arr=mandrill_0_cropped,
        kernel=kernel,
        win=None,
        boundary="symmetric",
        out_mode="valid",
        zoom=(1, 1),
        decimation="leading",
        axes=None,
        dtype=None,
        method="overlap_add",
)

# The output window does not depends on the out_mode and is always expressed in the coordinates of the full mode.
print(f"window with `out_mode`=\"full\" : {filtered_result_symmetric_full.window.tolist()}")
print(f"window with `out_mode`=\"same\" : {filtered_result_symmetric_same.window.tolist()}")
print(f"window with `out_mode`=\"valid\" : {filtered_result_symmetric_valid.window.tolist()}")